# Django 4 Templates — Comprehensive Learning Guide

This notebook consolidates all four modules of the Django 4 Templates course:

| Module | Title |
|--------|-------|
| 02 | Introduction to Django Templates |
| 03 | Discovering the Power of the Django Template System |
| 04 | Composing Templates to Build a Robust and Scalable Frontend |
| 05 | Debugging and Optimizing Django Templates |

**Course uses Django 4.2**

---
## Module 02 — Introduction to Django Templates

### Course Overview

- Overview of the templates
- Django Template System
- Inheritance and Composition
- Optimising Templates

### What is a Web Framework?

> A **web framework** is a set of tools and resources designed to simplify the web development process and make it easier to build a website.

Django is a high-level Python web framework that follows the **MVT (Model-View-Template)** architectural pattern.

### The MVT Architecture

The MVT pattern is divided into three parts:

| Component | Role |
|-----------|------|
| **Model** | Handles the application data (database layer) |
| **View** | Executes the business logic, carries and renders the data |
| **Template** | Presentation layer — HTML output + DTL syntax for dynamic content |

```
Django WebApp  ←→  URL  ←→  VIEW  ←→  MODEL  →  DATA
                               ↕
                           TEMPLATE
```

The flow:
1. Browser sends request to Django WebApp
2. URL dispatcher routes to the appropriate View
3. View fetches data from Model
4. View renders a Template with that data
5. Django returns the rendered HTML to the browser

### What are Django Templates?

Templates are the **representation layer** — the "T" component of the MVT pattern.

They:
- Control what users see on a webpage
- Render static content with HTML code
- Display dynamic data with **Django Template Language (DTL)**

The rendering pipeline:
```
TEMPLATE (</>  {{ }})  →  Interpreter  →  HTML (</>)
```

### Setting Up the Environment

#### Project structure for a Django project called `dailynews`

```
dailynews/
├── manage.py
├── dailynews/           ← project config package
│   ├── settings.py
│   ├── urls.py
│   └── wsgi.py
└── articles/            ← Django app
    ├── models.py
    ├── views.py
    ├── urls.py
    └── templates/
        └── articles/
            └── home.html
```

#### Setup commands

```bash
# Create virtual environment
python -m venv venv
source venv/bin/activate          # macOS/Linux
# venv\Scripts\activate           # Windows

# Install Django
pip install django

# Create project
django-admin startproject dailynews

# Create app
cd dailynews
python manage.py startapp articles

# Run server
python manage.py runserver
```

### URL Building: From URLs to Views

The MVT framework URL flow:

```
Django WebApp  →  URL  →  VIEW  →  TEMPLATE (rendered page)
               ←      ←       ←
```

#### Project-level `urls.py`

```python
# dailynews/urls.py
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('articles.urls')),
]
```

#### App-level `urls.py`

```python
# articles/urls.py
from django.urls import path
from . import views

urlpatterns = [
    path('', views.home, name='home'),
]
```

### Creating Your First Django Template

#### Step 1: The View

A view can return a raw `HttpResponse` or render a template:

```python
# articles/views.py
from django.shortcuts import render
from django.http import HttpResponse

# Option 1: Return raw HTML string
def home(request):
    html = "<html><body><h1>Daily News</h1></body></html>"
    return HttpResponse(html)

# Option 2: Render a template (recommended)
def home(request):
    return render(request, 'articles/home.html', {'title': 'Daily News'})
    #              ^request  ^template path        ^context dictionary
```

#### Step 2: The Template

The template uses the **Django Template Language (DTL)** to render dynamic content.

```html
<!-- articles/templates/articles/home.html -->
<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <title>Daily News</title>
  </head>
  <body>
    <header>
      <div class="title"><h1>{{ title }}</h1></div>
    </header>
  </body>
</html>
```

---
## Module 03 — Discovering the Power of the Django Template System

### Overview

- Django template built-in tags
- Built-in filters
- Adding static files

### The Django Template Syntax

DTL provides four types of constructs:

| Syntax | Purpose | Example |
|--------|---------|--------|
| `{{ variable }}` | Output a variable | `{{ title }}` |
| `{% tag %}` | Logic / control flow | `{% for item in list %}` |
| `{{ variable \| filter }}` | Transform output | `{{ name \| upper }}` |
| `{# comment #}` | Template comments | `{# This is hidden #}` |

### Built-in Template Tags

#### Variables

```html
<p>{{ Django }}</p>
```

#### Control Structures — `for` loop

```html
<html>
  <head><title>Daily News</title></head>
  <body>
    <div style="background:#2e2661">
      <h1>Daily News</h1>

      {% for result in results %}
        <div>{{ result }}</div>
      {% endfor %}

    </div>
  </body>
</html>
```

#### Control Structures — `if/else`

```html
{% if results %}
  <p>Found {{ results|length }} results</p>
{% else %}
  <p>No results found.</p>
{% endif %}
```

### Built-in Filters

Filters transform the displayed value of a variable. Syntax: `{{ variable | filter }}`

```html
<!-- upper: converts to uppercase -->
<p>{{ Django | upper }}</p>

<!-- lower: converts to lowercase -->
{% for result in results %}
  <div>{{ result.title | lower }}</div>
{% endfor %}

<!-- date: formats a date -->
<p>{{ result.createdAt | date }}</p>

<!-- length: returns the length of the value -->
<p>{{ results | length }}</p>

<!-- truncatechars: truncates text to N characters -->
<p>{{ result.abstract | truncatechars:100 }}</p>

<!-- safe: marks string as HTML-safe (disables auto-escaping) -->
<p>{{ html_content | safe }}</p>
```

### Making API Calls and Displaying Results

The course demo uses the **New York Times API** to fetch real news articles.

```python
# common/utils.py — helper to call the NYT API
import requests

def getResults(url):
    response = requests.get(url)
    data = response.json()
    return data.get('results', [])
```

```python
# articles/views.py
from django.shortcuts import render
from common.utils import getResults
import config

TOPSTORIES  = "https://api.nytimes.com/svc/topstories/v2/arts.json?api-key="
MOSTPOPULAR = "https://api.nytimes.com/svc/mostpopular/v2/emailed/7.json?api-key="
FEED        = "https://api.nytimes.com/svc/news/v3/content/all/all.json?api-key="

links = [
    {"name": "Home",           "path": "/"},
    {"name": "Top Stories",    "path": "/topstories"},
    {"name": "Most Popular",   "path": "/popular"},
    {"name": "Real Time Feed", "path": "/feed"},
]

apis = [
    {"name": "Top Stories API",  "path": "/topstories"},
    {"name": "Most Popular API", "path": "/popular"},
    {"name": "Times Wire API",   "path": "/feed"},
]

def results(request):
    pathname = str(request).split("/").pop(1)
    results = []
    subtitle = ""

    if pathname == "topstories":
        results = getResults(TOPSTORIES + config.api_key)
        subtitle = "Top"
    elif pathname == "popular":
        results = getResults(MOSTPOPULAR + config.api_key)
        subtitle = "Most Popular"
    elif pathname == "feed":
        results = getResults(FEED + config.api_key)
        subtitle = "Feed"

    return render(request, 'articles/results.html', {
        'title': 'Daily News',
        'subtitle': subtitle,
        'results': results,
        'links': links,
        'pathname': pathname + "/",
    })
```

### Adding and Managing Static Files

Static files (CSS, JS, images) are served using the `{% load static %}` tag.

```python
# settings.py
STATIC_URL = 'static/'
STATICFILES_DIRS = [
    BASE_DIR / 'static',
]
```

```html
{% load static %}
<!DOCTYPE html>
<html>
<head>
    <!-- Load a CSS file from static -->
    <link href="{% static 'css/styles.css' %}" rel="stylesheet" />

    <style>
        .bg {
            background: url("{% static 'img/background.jpg' %}") no-repeat fixed;
        }
    </style>
</head>
<body>
    <!-- Load an image -->
    <img src="{% static 'img/logo.png' %}" alt="Logo" />
</body>
</html>
```

---
## Module 04 — Composing Templates to Build a Robust and Scalable Frontend

### Overview

- Including templates in templates
- Creating reusable fragments
- Creating custom tags and filters
- Escaping user data

### Composition and Template Inheritance

> *"Template inheritance allows you to build a base 'skeleton' template that contains all the common elements of your site and defines blocks that child templates can override."*
> — Django Documentation

This is the most powerful feature of the Django template system — it allows you to **avoid repetition** (DRY principle) across pages.

Two mechanisms:

1. **`{% extends %}`** — Template inheritance (parent → child)
2. **`{% include %}`** — Insert a reusable fragment

### Template Inheritance: `base.html` + child templates

```html
<!-- static/templates/base.html — the parent "skeleton" -->
{% load static %}
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8" />
    <title>Daily News</title>
    <link href="{% static 'css/styles.css' %}" rel="stylesheet" />
    <style>
        .bg { background: url("{% static 'img/background.jpg' %}") 55% 70% no-repeat fixed; }
    </style>
</head>
<body>
    <div class="title bg">
        <h1>{{ title }}</h1>
    </div>

    {% block content %}
    {% endblock %}

</body>
</html>
```

```html
<!-- articles/templates/articles/home.html — child template -->
{% extends "base.html" %}
{% load static %}

{% block content %}

<div class="container">
  <div class="row mt-5 mx-1">
    {% for api in apis %}
      {% include 'articles/includes/home_card.html' with api=api %}
    {% endfor %}
  </div>
</div>

{% endblock %}
```

### Create Reusable Django Fragments

Fragments are small template files included with `{% include %}`. They promote reuse and keep templates DRY.

Typical fragment folder structure:
```
articles/templates/articles/
├── home.html
├── results.html
└── includes/
    ├── navigation.html
    ├── card.html
    ├── home_card.html
    └── count.html
```

```html
<!-- Basic include -->
{% include 'articles/includes/navigation.html' %}

<!-- Include with explicit variable -->
{% include 'articles/includes/home_card.html' with api=api %}

<!-- Include in a loop -->
{% for api in apis %}
  {% include 'articles/includes/home_card.html' with api=api %}
{% endfor %}

<!-- Include with only — restricts context to passed variables only -->
{% include 'fragment.html' with title=title only %}
```

### Creating Custom Template Tags and Filters

When built-in tags/filters are not enough, you can create your own.

**File location:** `articles/templatetags/custom_tags.py`  
The `templatetags/` directory must also contain an `__init__.py` file.

```python
# articles/templatetags/custom_tags.py
from django import template
from django.utils.html import mark_safe

register = template.Library()


@register.filter(is_safe=True, needs_autoescape=True)
def display_length(value, autoescape=True):
    """Returns an HTML string showing the count of results."""
    count = len(value)
    if count == 0:
        return mark_safe("<p class='text-right'>%s result</p>" % count)
    else:
        return mark_safe("<p class='text-right'>%s results</p>" % count)


@register.filter()
def display_active(value, arg):
    """Returns 'active' CSS class when the current path matches."""
    return 'active' if value == arg else ''


@register.filter()
def firstCap(value):
    """Capitalizes the first letter of every word."""
    result = ""
    for w in value.split(" "):
        result += w.capitalize() + " "
    return result


register.filter('display_length', display_length)
register.filter('display_active', display_active)
register.filter('firstCap', firstCap)
```

Using custom tags in a template:

```html
{% load custom_tags %}

<!-- display_length filter -->
{{ results | display_length }}

<!-- display_active filter: marks nav link as active -->
<a class="nav-link {{ pathname | display_active:link.path }}" href="{{ link.path }}">
    {{ link.name }}
</a>

<!-- firstCap filter -->
<p>{{ result.title | firstCap }}</p>
```

### Escaping User Data

Django auto-escapes HTML in templates by default to prevent **XSS (Cross-Site Scripting)** attacks.

| Mechanism | Description |
|-----------|-------------|
| Auto-escape (default) | Django escapes `<`, `>`, `&`, `"`, `'` automatically |
| `{% autoescape off %}` | Disable auto-escaping for a block |
| `{{ value \| safe }}` | Mark a single variable as safe (trusted HTML) |
| `mark_safe()` | Mark a Python string as safe in view/filter code |
| `{{ value \| escape }}` | Force-escape a variable |

```html
<!-- Auto-escaping: safe by default -->
<p>{{ user_input }}</p>  {# <script>alert('xss')</script> → rendered as plain text #}

<!-- Mark a specific variable as safe (trusted HTML) -->
<p>{{ html_content | safe }}</p>

<!-- Disable auto-escaping for a block -->
{% autoescape off %}
  {{ html_body }}
{% endautoescape %}

<!-- Force-escape even if safe-marked -->
<p>{{ value | escape }}</p>
```

```python
# In Python code: use mark_safe() carefully
from django.utils.html import mark_safe

# Only use mark_safe on strings you fully control
safe_html = mark_safe('<strong>Bold text</strong>')
```

---
## Module 05 — Debugging and Optimizing Django Templates

### Overview

- Debugging and tracking performances
- Caching and optimization
- Using a different template engine (Jinja2)

### Django Debug Toolbar

The **Django Debug Toolbar** is the standard tool for debugging and profiling Django applications.

#### Setup: 4 steps

**Step 1 — Install**

```bash
pip install django-debug-toolbar
```

**Steps 2–4 — `settings.py`**

```python
INSTALLED_APPS = [
    'articles',
    'debug_toolbar',          # Step 2: add the app
    'django.contrib.admin',
    # ...
]

MIDDLEWARE = [
    'debug_toolbar.middleware.DebugToolbarMiddleware',  # Step 3: add middleware (must be first)
    'django.middleware.security.SecurityMiddleware',
    # ...
]

# Step 4: toolbar only appears for these IPs
INTERNAL_IPS = [
    '127.0.0.1',
]

DEBUG = True  # Must be True for toolbar to appear
```

**Wire up toolbar URLs in `urls.py`**

```python
# dailynews/urls.py
from django.conf import settings
import debug_toolbar

urlpatterns = [
    # ...
]

if settings.DEBUG:
    urlpatterns = [
        path('__debug__/', include(debug_toolbar.urls)),
    ] + urlpatterns
```

### Caching and Optimisation

> *"To cache something is to save the result of an expensive calculation so that you don't have to perform the calculation next time."*

Caching is especially valuable for views that make external API calls or run complex database queries.

#### Cache backends in Django

| Backend | Use Case |
|---------|----------|
| `LocMemCache` | Per-process in-memory cache (development) |
| `FileBasedCache` | Cache to filesystem |
| `MemcachedCache` | Distributed cache (production) |
| `RedisCache` | Redis-based cache (production) |

#### Configure cache backend in `settings.py`

```python
CACHES = {
    'default': {
        'BACKEND': 'django.core.cache.backends.locmem.LocMemCache',
        'LOCATION': '127.0.0.1:11211',
    }
}
```

#### Cache an entire view with `@cache_page`

```python
# articles/views.py
from django.views.decorators.cache import cache_page

@cache_page(600)  # cache for 600 seconds (10 minutes)
def home(request):
    return render(request, 'articles/home.html', {'title': 'Daily News', 'apis': apis})
```

#### Manual low-level caching

```python
from django.core.cache import cache

def home(request):
    results = cache.get('top_stories')
    if results is None:
        results = getResults(TOPSTORIES + config.api_key)
        cache.set('top_stories', results, timeout=600)
    return render(request, 'articles/home.html', {'results': results})
```

#### Template fragment caching

```html
{% load cache %}

{% cache 600 sidebar %}
  <!-- This block is cached for 600 seconds under the key 'sidebar' -->
  {% include 'articles/includes/navigation.html' %}
{% endcache %}
```

### Using a Different Template Engine — Jinja2

Django supports multiple template engines. **Jinja2** is a popular alternative to DTL, used in Flask and other frameworks.

#### Key differences between DTL and Jinja2

| Feature | DTL | Jinja2 |
|---------|-----|--------|
| Variable | `{{ var }}` | `{{ var }}` |
| Tag | `{% tag %}` | `{% tag %}` |
| Filter | `{{ var \| filter }}` | `{{ var \| filter }}` |
| Function calls | Not allowed | Allowed: `{{ func() }}` |
| Macros | No | Yes |
| Auto-escape | Optional | Enabled by default |
| Performance | Good | Generally faster |

#### Configure both DTL and Jinja2 in `settings.py`

```python
TEMPLATES = [
    {
        # Django Template Language (DTL)
        'BACKEND': 'django.template.backends.django.DjangoTemplates',
        'DIRS': [
            BASE_DIR / 'static/templates'
        ],
        'APP_DIRS': True,
        'OPTIONS': {
            'context_processors': [
                'django.template.context_processors.debug',
                'django.template.context_processors.request',
                'django.contrib.auth.context_processors.auth',
                'django.contrib.messages.context_processors.messages',
            ],
        },
    },
    {
        # Jinja2 engine
        'BACKEND': 'django.template.backends.jinja2.Jinja2',
        'DIRS': [
            BASE_DIR / 'static/templates',
            BASE_DIR / 'jinja2/',
        ],
    },
]
```

#### Custom Jinja2 environment (optional)

```python
# dailynews/jinja2.py
from jinja2 import Environment
from django.templatetags.static import static
from django.urls import reverse

def environment(**options):
    env = Environment(**options)
    env.globals.update({
        'static': static,
        'url': reverse,
    })
    return env
```

#### A Jinja2 template

```html
<!-- jinja2/articles/index.html -->
<!DOCTYPE html>
<html>
<head>
    <title>{{ title }}</title>
    <link rel="stylesheet" href="{{ static('css/styles.css') }}">
</head>
<body>
    {% for result in results %}
        <div>{{ result.title | upper }}</div>
    {% endfor %}
</body>
</html>
```

---
## Quick Reference Cheat Sheet

```
╔══════════════════════════════════════════════════════════════════════╗
║              DJANGO TEMPLATE LANGUAGE (DTL) CHEAT SHEET              ║
╠══════════════════════════════════════════════════════════════════════╣
║  SYNTAX                                                              ║
║  {{ variable }}            Output a variable                         ║
║  {% tag %}  {% endtag %}   Control flow / logic                      ║
║  {{ var | filter }}        Transform a value                         ║
║  {# comment #}             Template comment (not rendered)           ║
╠══════════════════════════════════════════════════════════════════════╣
║  CONTROL TAGS                                                        ║
║  {% for x in list %}  {% endfor %}                                   ║
║  {% if cond %}  {% elif cond %}  {% else %}  {% endif %}             ║
║  {% block name %}  {% endblock %}    (inheritance)                   ║
║  {% extends 'base.html' %}           (inherit from parent)           ║
║  {% include 'fragment.html' %}       (insert fragment)               ║
║  {% load static %}                   (enable static tag)             ║
║  {% load custom_tags %}              (enable custom tags)            ║
║  {% static 'path/to/file.css' %}     (generate static URL)           ║
║  {% url 'view-name' %}               (generate URL)                  ║
║  {% cache 600 key %}  {% endcache %}  (cache a fragment)             ║
║  {% autoescape off %}  {% endautoescape %}  (disable auto-escape)    ║
╠══════════════════════════════════════════════════════════════════════╣
║  COMMON BUILT-IN FILTERS                                             ║
║  upper / lower           Case conversion                             ║
║  length                  Count items                                 ║
║  truncatechars:N         Truncate to N chars                         ║
║  date / time             Format date/time                            ║
║  safe                    Mark as trusted HTML                        ║
║  escape                  Force HTML-escape                           ║
║  default:'fallback'      Use fallback if value is falsy              ║
║  first / last            Get first/last item of a list               ║
║  join:', '               Join list items with separator              ║
║  add:N                   Add N to the value                          ║
╠══════════════════════════════════════════════════════════════════════╣
║  SPECIAL FOR LOOP VARIABLES                                          ║
║  {{ forloop.counter }}     1-indexed iteration count                 ║
║  {{ forloop.counter0 }}    0-indexed iteration count                 ║
║  {{ forloop.revcounter }}  Iterations remaining                      ║
║  {{ forloop.first }}       True on first iteration                   ║
║  {{ forloop.last }}        True on last iteration                    ║
╠══════════════════════════════════════════════════════════════════════╣
║  PERFORMANCE TOOLS                                                   ║
║  pip install django-debug-toolbar                                    ║
║  @cache_page(seconds)     Cache entire view                          ║
║  cache.get/set()          Manual low-level caching                   ║
║  {% cache N key %}        Cache a template fragment                  ║
╚══════════════════════════════════════════════════════════════════════╝
```

---
## Course Summary

| Topic | Key Takeaways |
|-------|---------------|
| **MVT Architecture** | Templates are the presentation layer; views pass context dicts |
| **DTL Syntax** | `{{ var }}`, `{% tag %}`, `{{ var\|filter }}`, `{# comment #}` |
| **Built-in Tags** | `for`, `if`, `block`, `extends`, `include`, `static`, `url` |
| **Built-in Filters** | `upper`, `lower`, `length`, `date`, `safe`, `truncatechars` |
| **Template Inheritance** | `base.html` with `{% block %}` + child with `{% extends %}` |
| **Reusable Fragments** | `{% include %}` with partial template files |
| **Custom Tags/Filters** | `templatetags/` package with `register = template.Library()` |
| **Security** | Auto-escaping on by default; use `\| safe` / `mark_safe` carefully |
| **Debug Toolbar** | Install, add app + middleware, set `INTERNAL_IPS`, `DEBUG=True` |
| **Caching** | `@cache_page`, `cache.get/set`, `{% cache %}` fragment caching |
| **Jinja2** | Alternative engine — configure in `TEMPLATES` with separate `DIRS` |